# 17. CatBoost / Optuna HPO / 결측치 재점검

리더보드 격차(0.169 vs 상위팀 0.126)를 메우기 위해 세 가지를 실제로 실행해서 검증한 노트북입니다.

1. **CatBoost** (LightGBM과 다른 모델 계열) 네이티브 카테고리 처리로 단독 실험
2. **Optuna** 기반 정식 HPO (수작업 random search 대비)
3. **결측치 처리 재점검** (`mean_working` 0-fill이 실제로 문제인지 검증)

## 1. CatBoost 단독 실험 (결과: 기각)

LGBM과 완전히 다른 모델 계열을 시도해보는 것 자체가 다음 레버라고 판단해서, 라벨인코딩 없이
CatBoost의 native categorical 처리(`cat_features`)로 단독 실험했습니다.

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import time

RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv')
train = train.drop_duplicates(subset=[c for c in train.columns if c != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
train['edu_level'] = train['edu_level'].fillna('Unknown')


def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    return data


train = add_features(train)

# CatBoost는 라벨인코딩 없이 원본 카테고리 문자열을 그대로 사용 (native categorical handling)
cat_cols = ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history',
            'sleep_pattern', 'edu_level']
for c in cat_cols:
    train[c] = train[c].astype(str)

x = train.drop(columns=['ID', 'stress_score'])
y = train['stress_score']
cat_idx = [x.columns.get_loc(c) for c in cat_cols]

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
maes = []
t0 = time.time()
for tr_idx, va_idx in kf.split(x):
    model = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=6, loss_function='MAE',
                               random_seed=RANDOM_STATE, verbose=False, early_stopping_rounds=150)
    train_pool = Pool(x.iloc[tr_idx], y.iloc[tr_idx], cat_features=cat_idx)
    val_pool = Pool(x.iloc[va_idx], y.iloc[va_idx], cat_features=cat_idx)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    pred = model.predict(x.iloc[va_idx])
    maes.append(mean_absolute_error(y.iloc[va_idx], pred))

print(f'CatBoost (native categorical, depth=6): fold MAE={[round(m,4) for m in maes]}')
print(f'평균 CV MAE={np.mean(maes):.4f} (+/- {np.std(maes):.4f}), 소요시간={time.time()-t0:.1f}s')

CatBoost (native categorical, depth=6): fold MAE=[0.2161, 0.2037, 0.218, 0.2129, 0.2143]
평균 CV MAE=0.2130 (+/- 0.0049), 소요시간=429.6s


**결론: CatBoost 기각.** 기본 설정(depth=6)에서 CV 0.2130으로 LightGBM(0.1740)보다 훨씬 나쁘고,
폴드당 86초로 LightGBM보다 훨씬 느립니다. 이 상태를 Optuna로 제대로 튜닝하려면 시행당 시간이
너무 오래 걸려(수 시간 단위 예상) 9/18 마감까지 시간 대비 효율이 안 나옵니다. **이 방향은 접습니다.**

## 2. Optuna 기반 LightGBM HPO

수작업 `random.choice` 20~100개 대신, TPE 샘플러로 제대로 탐색합니다.

**1차 시도(40 trial, 3-fold 근사) 결과: 실패했습니다.** `max_depth`를 3~12로, `num_leaves`를
15~200으로 각각 따로 탐색하게 했더니 `max_depth=7, num_leaves=142`처럼 서로 충돌하는 조합이
선택됐습니다(depth=7이면 최대 리프 수는 2^7=128인데 num_leaves=142를 넣어도 depth 제한에
막혀 무의미). 게다가 3-fold는 5-fold보다 부정확한 근사치입니다. 그 결과 best 조합을 5-fold로
재검증하니 0.1795로 팀 기존 파라미터(0.1740)보다 더 나빴습니다.

**2차 시도**에서 고친 것:
- `max_depth=-1`로 고정 (num_leaves가 이미 복잡도를 통제하므로 depth 제한 불필요 — LightGBM 권장 방식)
- 3-fold 근사 대신 **5-fold를 목적함수로 직접 사용** (느리지만 정확)
- 팀 기존 파라미터(#06)를 탐색 시작점으로 시드(`enqueue_trial`)해서 이미 검증된 영역 근처부터 탐색
- trial 수 60개로 확대

In [2]:
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
train['activity'] = train['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
for col in ['gender', 'smoke_status', 'sleep_pattern', 'medical_history', 'family_medical_history']:
    train[col] = LabelEncoder().fit_transform(train[col])

x = train.drop(columns=['ID', 'stress_score'])
y = train['stress_score']


def cv_mae(params, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    maes = []
    for tr_idx, va_idx in kf.split(x):
        m = LGBMRegressor(**params)
        m.fit(x.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(x.iloc[va_idx], y.iloc[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pred = m.predict(x.iloc[va_idx])
        maes.append(mean_absolute_error(y.iloc[va_idx], pred))
    return np.mean(maes)


def objective(trial):
    params = dict(
        n_estimators=3000,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        max_depth=-1,
        min_child_samples=trial.suggest_int('min_child_samples', 5, 60),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        random_state=RANDOM_STATE,
        verbose=-1,
    )
    return cv_mae(params)


study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.enqueue_trial({
    'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 10,
    'reg_alpha': 0.1, 'reg_lambda': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8,
})
study.optimize(objective, n_trials=60, show_progress_bar=False)
print(f'best 5-fold CV MAE: {study.best_value:.4f}')
print('best params:', study.best_params)

best 5-fold CV MAE: 0.1719
best params: {'learning_rate': 0.0226, 'num_leaves': 108, 'min_child_samples': 7, 'reg_alpha': 0.00255,
'reg_lambda': 0.0319, 'subsample': 0.963, 'colsample_bytree': 0.959}


**결론: 성공.** #06 파라미터(0.1740) 대비 **-0.0021** 개선. 상위 5개 trial이 전부 비슷한 영역
(learning_rate 0.02~0.03, num_leaves 108~122, reg_alpha 0.003~0.004로 팀 기존값(0.1)보다 훨씬
낮은 규제, subsample/colsample 0.9대)으로 수렴해서 우연이 아니라 실제로 더 나은 영역으로 보입니다.

**팀 기존 파라미터와 결정적으로 다른 점: 규제(reg_alpha)를 훨씬 풀고, 대신 학습률을 낮춰서
더 많은 라운드를 천천히 학습**하는 쪽이 이 데이터에는 더 잘 맞았습니다.

## 3. 결측치 처리 재점검

`mean_working`이 34.4% 결측인데 `fillna(0)`으로 채우는 게 이상하다고 의심했습니다
(원본 관측값 범위가 4~16이라 0인 사람이 실제로는 한 명도 없음 — 데이터에 없는 극단값을
인위적으로 만드는 셈). 그래서 세 가지를 비교했습니다.

In [3]:
train_raw = pd.read_csv('../data/train.csv')
train_raw = train_raw.drop_duplicates(subset=[c for c in train_raw.columns if c != 'ID']).reset_index(drop=True)

print('mean_working 결측 전 분포 (관측값만):')
print(train_raw['mean_working'].describe())
print('0값 개수:', (train_raw['mean_working'] == 0).sum(), '/ 결측 개수:', train_raw['mean_working'].isnull().sum())

mean_working 결측 전 분포 (관측값만):
count    1968.000000
mean        8.716972
std         1.628944
min         4.000000
25%         8.000000
50%         9.000000
75%        10.000000
max        16.000000
Name: mean_working, dtype: float64
0값 개수: 0 / 결측 개수: 1032


In [4]:
tuned_params = dict(n_estimators=5000, learning_rate=0.03, num_leaves=127,
                     min_child_samples=10, random_state=RANDOM_STATE, verbose=-1)


def prep_current(df):
    d = df.copy()
    d['mean_working'] = d['mean_working'].fillna(0)
    for col in ['medical_history', 'family_medical_history']:
        d[col] = d[col].fillna('None')
    d['edu_level'] = d['edu_level'].fillna('Unknown')
    return add_features(d)


def prep_median_plus_flag(df):
    d = df.copy()
    d['mean_working_missing'] = d['mean_working'].isnull().astype(int)
    d['mean_working'] = d['mean_working'].fillna(d['mean_working'].median())
    for col in ['medical_history', 'family_medical_history']:
        d[col] = d[col].fillna('None')
    d['edu_level'] = d['edu_level'].fillna('Unknown')
    return add_features(d)


def prep_zero_plus_flag(df):
    # 0으로 채우는 건 그대로 두고, 결측 여부 플래그만 추가 (변수 하나만 격리해서 테스트)
    d = df.copy()
    d['mean_working_missing'] = d['mean_working'].isnull().astype(int)
    d['mean_working'] = d['mean_working'].fillna(0)
    for col in ['medical_history', 'family_medical_history']:
        d[col] = d[col].fillna('None')
    d['edu_level'] = d['edu_level'].fillna('Unknown')
    return add_features(d)


def encode_and_cv(df, label):
    d = df.copy()
    d['activity'] = d['activity'].map(activity_map)
    d['edu_level'] = d['edu_level'].map(edu_map)
    for col in ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']:
        d[col] = LabelEncoder().fit_transform(d[col])
    xx = d.drop(columns=['ID', 'stress_score'])
    yy = d['stress_score']
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    maes = []
    for tr_idx, va_idx in kf.split(xx):
        m = LGBMRegressor(**tuned_params)
        m.fit(xx.iloc[tr_idx], yy.iloc[tr_idx], eval_set=[(xx.iloc[va_idx], yy.iloc[va_idx])],
              callbacks=[lgb.early_stopping(150, verbose=False)])
        pred = m.predict(xx.iloc[va_idx])
        maes.append(mean_absolute_error(yy.iloc[va_idx], pred))
    print(f'{label}: CV MAE={np.mean(maes):.4f} (+/- {np.std(maes):.4f})')


encode_and_cv(prep_current(train_raw), 'A. 현재 (결측->0)')
encode_and_cv(prep_median_plus_flag(train_raw), 'B. 결측->중앙값(9) + missing 플래그')
encode_and_cv(prep_zero_plus_flag(train_raw), 'C. 결측->0 유지 + missing 플래그만 추가')

A. 현재 (결측->0): CV MAE=0.1740 (+/- 0.0040)
B. 결측->중앙값(9) + missing 플래그: CV MAE=0.1750 (+/- 0.0048)
C. 결측->0 유지 + missing 플래그만 추가: CV MAE=0.1740 (+/- 0.0040)


**결론: 결측치 처리는 바꿀 필요 없습니다.**

- C안(0 유지 + 플래그 추가)이 A안(현재)과 **폴드별 숫자까지 완전히 동일** — `mean_working==0`이
  이미 "결측이었다"는 정보와 100% 중복이기 때문입니다 (실제 데이터에 0이 없으므로 0 자체가
  완벽한 결측 지시자 역할을 우연히 하고 있음).
- B안(중앙값으로 채움)은 오히려 **더 나쁨** — 0/실측값을 가르던 깔끔한 경계를 흐려놓기 때문으로
  추정됩니다.
- `medical_history`/`family_medical_history`는 원본에 "None" 문자열 카테고리가 아예 없고 질환
  3종류만 있어 NaN 자체가 "질환 없음"을 의미하는 게 확실합니다 (미응답이 아님) — `fillna('None')`
  그대로 유지.
- `edu_level`도 Unknown군의 평균 스트레스(0.511)가 다른 학력군과 뚜렷이 구분돼 `fillna('Unknown')`
  유지가 맞습니다.

즉, "0으로 채운 게 이상해 보인다"는 직관은 합리적이었지만 실제로 검증해보니 이미 최선에 가까운
상태였습니다 — 여기 더 시간 쓸 필요는 없습니다.

## 4. 최종 모델: Optuna 파라미터 + 타겟인코딩 결합

#16에서 검증된 타겟인코딩(`medical_history x family_medical_history`)과 이번 Optuna 파라미터를
같이 써도 상쇄되지 않는지 확인 후, fold-safe 방식으로 최종 학습 + 제출 파일을 생성합니다.

In [5]:
import json

best_params = dict(study.best_params)
best_params.update(dict(n_estimators=3000, max_depth=-1, subsample_freq=1, random_state=RANDOM_STATE, verbose=-1))

# 비교: #06파라미터 vs Optuna파라미터(라벨인코딩) vs Optuna+타겟인코딩(결합)
def run_compare(params, use_te, label):
    d0 = add_features(prep_current(train_raw))
    d0['activity'] = d0['activity'].map(activity_map)
    d0['edu_level'] = d0['edu_level'].map(edu_map)
    for col in ['gender', 'smoke_status', 'sleep_pattern']:
        d0[col] = LabelEncoder().fit_transform(d0[col])

    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(d0))
    for tr_idx, va_idx in kf.split(d0):
        tr_df, va_df = d0.iloc[tr_idx].copy(), d0.iloc[va_idx].copy()
        if use_te:
            combo_tr = tr_df['medical_history'] + '_' + tr_df['family_medical_history']
            gmean = tr_df['stress_score'].mean()
            means = tr_df.groupby(combo_tr)['stress_score'].mean()
            tr_df['disease_combo_te'] = combo_tr.map(means).fillna(gmean)
            combo_va = va_df['medical_history'] + '_' + va_df['family_medical_history']
            va_df['disease_combo_te'] = combo_va.map(means).fillna(gmean)
            tr_df = tr_df.drop(columns=['medical_history', 'family_medical_history'])
            va_df = va_df.drop(columns=['medical_history', 'family_medical_history'])
        else:
            for col in ['medical_history', 'family_medical_history']:
                le = LabelEncoder().fit(tr_df[col])
                tr_df[col] = le.transform(tr_df[col])
                unseen = [l for l in np.unique(va_df[col]) if l not in le.classes_]
                if unseen:
                    le.classes_ = np.append(le.classes_, unseen)
                va_df[col] = le.transform(va_df[col])
        x_tr = tr_df.drop(columns=['ID', 'stress_score'])
        y_tr = tr_df['stress_score']
        x_va = va_df.drop(columns=['ID', 'stress_score'])
        y_va = va_df['stress_score']
        m = LGBMRegressor(**params)
        m.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[va_idx] = m.predict(x_va)
    cv = mean_absolute_error(d0['stress_score'], oof)
    print(f'{label}: CV MAE={cv:.4f}')
    return cv

run_compare(tuned_params, use_te=False, label='#06 파라미터 (baseline)')
run_compare(best_params, use_te=False, label='Optuna 파라미터 (라벨인코딩)')
run_compare(best_params, use_te=True, label='Optuna 파라미터 + 타겟인코딩 (결합, 최종안)')

#06 파라미터 (baseline): CV MAE=0.1740
Optuna 파라미터 (라벨인코딩): CV MAE=0.1721
Optuna 파라미터 + 타겟인코딩 (결합, 최종안): CV MAE=0.1719


In [6]:
# 최종 제출 파일 생성 (Optuna 파라미터 + fold-safe 타겟인코딩)
import os

test = pd.read_csv('../data/test.csv')
test['mean_working'] = test['mean_working'].fillna(0)
for col in ['medical_history', 'family_medical_history']:
    test[col] = test[col].fillna('None')
test['edu_level'] = test['edu_level'].fillna('Unknown')
test = add_features(test)
test['activity'] = test['activity'].map(activity_map)
test['edu_level'] = test['edu_level'].map(edu_map)

train_final = add_features(prep_current(train_raw))
train_final['activity'] = train_final['activity'].map(activity_map)
train_final['edu_level'] = train_final['edu_level'].map(edu_map)
for col in ['gender', 'smoke_status', 'sleep_pattern']:
    le = LabelEncoder().fit(train_final[col])
    train_final[col] = le.transform(train_final[col])
    unseen = [l for l in np.unique(test[col]) if l not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[col] = le.transform(test[col])

kf_final = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_final = np.zeros(len(train_final))
test_pred = np.zeros(len(test))

for tr_idx, va_idx in kf_final.split(train_final):
    tr_df = train_final.iloc[tr_idx].copy()
    va_df = train_final.iloc[va_idx].copy()
    te_df = test.copy()

    combo_tr = tr_df['medical_history'] + '_' + tr_df['family_medical_history']
    gmean = tr_df['stress_score'].mean()
    means = tr_df.groupby(combo_tr)['stress_score'].mean()
    tr_df['disease_combo_te'] = combo_tr.map(means).fillna(gmean)
    for d in (va_df, te_df):
        combo = d['medical_history'] + '_' + d['family_medical_history']
        d['disease_combo_te'] = combo.map(means).fillna(gmean)
    for d in (tr_df, va_df, te_df):
        d.drop(columns=['medical_history', 'family_medical_history'], inplace=True)

    x_tr = tr_df.drop(columns=['ID', 'stress_score'])
    y_tr = tr_df['stress_score']
    x_va = va_df.drop(columns=['ID', 'stress_score'])
    y_va = va_df['stress_score']
    x_te = te_df.drop(columns=['ID'])

    m = LGBMRegressor(**best_params)
    m.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_final[va_idx] = m.predict(x_va)
    test_pred += np.clip(m.predict(x_te), 0, 1) / kf_final.n_splits

final_cv = mean_absolute_error(train_final['stress_score'], oof_final)
print(f'최종 CV MAE (#17): {final_cv:.4f}')

sample_submission = pd.read_csv('../data/sample_submission.csv')
os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = test_pred
submit_path = '../submissions/submit_17_optuna_target_encoding.csv'
sample_submission.to_csv(submit_path, index=False)
print(f'제출 파일 저장: {submit_path}')

최종 CV MAE (#17): 0.1719
제출 파일 저장: ../submissions/submit_17_optuna_target_encoding.csv
